# DSPy Entity Extraction Example

This notebook demonstrates how DSPy can extract named entities (persons, locations, organizations) from text with minimal code.

In [1]:
import dspy

In [2]:
# Configure DSPy with llama model
lm = dspy.LM("ollama_chat/llama3.2:latest", api_base="http://localhost:11434", api_key="")
dspy.configure(lm=lm)

# Disable caching 
dspy.configure_cache(enable_memory_cache=False, enable_disk_cache=False)

In [8]:
# Simple version

simple_model = dspy.Predict("text->persons, locations, organizations")

In [9]:
output = simple_model(text="TechCorp's CEO Sarah Chen met with DataSystems' Michael Torres in Riverside to discuss cloud computing.")

In [10]:
print(output)

Prediction(
    persons='Sarah Chen\nMichael Torres',
    locations='Riverside',
    organizations='TechCorp\nDataSystems'
)


In [11]:
system_prompt = simple_model.inspect_history(n=1)
print(system_prompt)





[2026-02-14T23:02:25.761456]

System message:

Your input fields are:
1. `text` (str):
Your output fields are:
1. `persons` (str): 
2. `locations` (str): 
3. `organizations` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## persons ## ]]
{persons}

[[ ## locations ## ]]
{locations}

[[ ## organizations ## ]]
{organizations}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `text`, produce the fields `persons`, `locations`, `organizations`.


User message:

[[ ## text ## ]]
TechCorp's CEO Sarah Chen met with DataSystems' Michael Torres in Riverside to discuss cloud computing.

Respond with the corresponding output fields, starting with the field `[[ ## persons ## ]]`, then `[[ ## locations ## ]]`, then `[[ ## organizations ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## persons ## ]]
Sarah Chen
Michael Torres



## Step 1: Define the Signature

A signature defines the input-output behavior we want

In [12]:
class EntityExtraction(dspy.Signature):
    """Extract named entities (persons, locations, organizations) from the given text."""
    
    text: str = dspy.InputField(desc="The text to extract entities from")
    persons: list[str] = dspy.OutputField(desc="List of person names extracted from the text")
    locations: list[str] = dspy.OutputField(desc="List of location names extracted from the text")
    organizations: list[str] = dspy.OutputField(desc="List of organization names extracted from the text")

## Step 2: Create a Module

Wrap the signature in a module (using ChainOfThought for better reasoning)

In [13]:
entity_extractor_predictor = dspy.Predict(EntityExtraction, cache=False)

In [14]:
# Create the entity extractor
entity_extractor_cot = dspy.ChainOfThought(EntityExtraction, cache=False)

## Step 3: Test on Example Text

In [15]:
# Example text
text = "TechCorp's CEO Sarah Chen met with DataSystems' Michael Torres in Riverside to discuss cloud computing."

# Extract entities
result_from_predictor = entity_extractor_predictor(text=text)

print("Result from predictor:")
print("Persons:", result_from_predictor.persons)
print("Locations:", result_from_predictor.locations)
print("Organizations:", result_from_predictor.organizations)

Result from predictor:
Persons: ['Sarah Chen', 'Michael Torres']
Locations: ['Riverside']
Organizations: ['TechCorp', 'DataSystems']


In [16]:
print(entity_extractor_predictor.inspect_history(n=1))





[2026-02-14T23:07:38.837830]

System message:

Your input fields are:
1. `text` (str): The text to extract entities from
Your output fields are:
1. `persons` (list[str]): List of person names extracted from the text
2. `locations` (list[str]): List of location names extracted from the text
3. `organizations` (list[str]): List of organization names extracted from the text
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## persons ## ]]
{persons}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## locations ## ]]
{locations}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## organizations ## ]]
{organizations}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## completed ## ]]
In adhering to this 

In [17]:
result_from_cot = entity_extractor_cot(text=text)

print("Result from COT:")
print("Persons:", result_from_cot.persons)
print("Locations:", result_from_cot.locations)
print("Organizations:", result_from_cot.organizations)

Result from COT:
Persons: ['Sarah Chen', 'Michael Torres']
Locations: ['Riverside']
Organizations: ['TechCorp', 'DataSystems']


In [21]:
# inspect the prompt 
system_prompt_cot = entity_extractor_cot.inspect_history(n=1)
print(system_prompt_cot)





[2026-02-15T16:23:33.617853]

System message:

Your input fields are:
1. `text` (str): The text to extract entities from
Your output fields are:
1. `reasoning` (str): 
2. `persons` (list[str]): List of person names extracted from the text
3. `locations` (list[str]): List of location names extracted from the text
4. `organizations` (list[str]): List of organization names extracted from the text
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "persons": "{persons}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"array\", \"items\": {\"type\": \"string\"}}",
  "locations": "{locations}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"array\", \"items\": {\"type\": \"string\"}}",
  "organizations": "{organizations}

## Step 4: Inspect the Prompt

See what DSPy is actually sending to the LM

In [ ]:
# View the last prompt sent to the LM
lm.inspect_history(n=1)





[2026-02-15T16:23:31.875668]

System message:

Your input fields are:
1. `text` (str): The text to extract entities from
Your output fields are:
1. `reasoning` (str): 
2. `persons` (list[str]): List of person names extracted from the text
3. `locations` (list[str]): List of location names extracted from the text
4. `organizations` (list[str]): List of organization names extracted from the text
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## reasoning ## ]]
{reasoning}

[[ ## persons ## ]]
{persons}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## locations ## ]]
{locations}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## organizations ## ]]
{organizations}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"typ

## Step 5: Optimize with Training Data

Create training examples and use DSPy's optimizer to improve performance

In [27]:
# Create training examples
train_examples = [
    dspy.Example(
        text="Robert Martinez founded AeroTech in Riverside, California and later moved to Nevada.",
        persons=["Robert Martinez"],
        locations=["Riverside, California", "Nevada"],
        organizations=["AeroTech"]
    ).with_inputs("text"),
    dspy.Example(
        text="The Global Trade Alliance held a meeting in Portland with Director-General Lisa Wong.",
        persons=["Lisa Wong"],
        locations=["Portland"],
        organizations=["Global Trade Alliance"]
    ).with_inputs("text"),
    dspy.Example(
        text="TechCorp's Sarah Chen met with DataSystems' Michael Torres in Riverside to discuss cloud computing.",
        persons=["Sarah Chen", "Michael Torres"],
        locations=["Riverside"],
        organizations=["TechCorp", "DataSystems"]
    ).with_inputs("text"),
]

# Define a simple metric
def entity_metric(example, pred, trace=None):
    # Simple overlap metric - in practice, you'd want something more sophisticated
    predicted_persons = set(pred.persons.split(","))
    predicted_locations = set(pred.locations.split(","))
    predicted_organizations = set(pred.organizations.split(","))

    example_persons = set(example.persons.split(","))
    example_locations = set(example.locations.split(","))
    example_organizations = set(example.organizations.split(","))

    # Calculate precision for each entity type
    intersection_persons = predicted_persons.intersection(example_persons)
    intersection_locations = predicted_locations.intersection(example_locations)
    intersection_organizations = predicted_organizations.intersection(example_organizations)

    # Average precision across all entity types
    overlap_persons = len(intersection_persons) / len(predicted_persons) if predicted_persons else 0.0
    overlap_locations = len(intersection_locations) / len(predicted_locations) if predicted_locations else 0.0
    overlap_organizations = len(intersection_organizations) / len(predicted_organizations) if predicted_organizations else 0.0
    
    overall_overlap = (overlap_persons + overlap_locations + overlap_organizations) / 3.0
    return overall_overlap

def entities_f1_metric(ground_truth, pred, trace=None):
    # Simple F1 metric - calculates precision, recall, and F1 for each entity type
    predicted_persons = set(pred.persons)
    predicted_locations = set(pred.locations)
    predicted_organizations = set(pred.organizations)

    ground_truth_persons = set(ground_truth.persons)
    ground_truth_locations = set(ground_truth.locations)
    ground_truth_organizations = set(ground_truth.organizations)

    # Calculate intersection for each entity type
    intersection_persons = predicted_persons.intersection(ground_truth_persons)
    intersection_locations = predicted_locations.intersection(ground_truth_locations)
    intersection_organizations = predicted_organizations.intersection(ground_truth_organizations)

    # Calculate F1 for persons
    precision_persons = len(intersection_persons) / len(predicted_persons) if predicted_persons else 0.0
    recall_persons = len(intersection_persons) / len(ground_truth_persons) if ground_truth_persons else 0.0
    f1_persons = 2 * (precision_persons * recall_persons) / (precision_persons + recall_persons) if (precision_persons + recall_persons) > 0 else 0.0
    
    # Calculate F1 for locations
    precision_locations = len(intersection_locations) / len(predicted_locations) if predicted_locations else 0.0
    recall_locations = len(intersection_locations) / len(ground_truth_locations) if ground_truth_locations else 0.0
    f1_locations = 2 * (precision_locations * recall_locations) / (precision_locations + recall_locations) if (precision_locations + recall_locations) > 0 else 0.0
    
    # Calculate F1 for organizations
    precision_organizations = len(intersection_organizations) / len(predicted_organizations) if predicted_organizations else 0.0
    recall_organizations = len(intersection_organizations) / len(ground_truth_organizations) if ground_truth_organizations else 0.0
    f1_organizations = 2 * (precision_organizations * recall_organizations) / (precision_organizations + recall_organizations) if (precision_organizations + recall_organizations) > 0 else 0.0
    
    # Average F1 across all entity types (macro F1)
    overall_f1 = (f1_persons + f1_locations + f1_organizations) / 3.0
    return overall_f1



In [ ]:
# We will be using the BootstrapFewShot optimizer
from dspy.teleprompt import BootstrapFewShot

bootstrap_optimizer = BootstrapFewShot(metric=entities_f1_metric, max_bootstrapped_demos=2, max_rounds=5, metric_threshold=0.8)
optimized_extractor = bootstrap_optimizer.compile(entity_extractor_predictor, trainset=train_examples)

print("Optimization complete!")

100%|██████████| 3/3 [00:04<00:00,  1.35s/it]

Bootstrapped 2 full traces after 2 examples for up to 5 rounds, amounting to 7 attempts.
✓ Optimization complete!


In [39]:
# Access the demos that were added to the predictor
if hasattr(optimized_extractor, 'demos'):
    print(f"Number of demos: {len(optimized_extractor.demos)}")
    for i, demo in enumerate(optimized_extractor.demos):
        print(f"\nDemo {i+1}:")
        print(demo)

Number of demos: 3

Demo 1:
Example({'augmented': True, 'text': 'The Global Trade Alliance held a meeting in Portland with Director-General Lisa Wong.', 'persons': ['Lisa Wong'], 'locations': ['Portland'], 'organizations': ['Global Trade Alliance']}) (input_keys=None)

Demo 2:
Example({'augmented': True, 'text': "TechCorp's Sarah Chen met with DataSystems' Michael Torres in Riverside to discuss cloud computing.", 'persons': ['Sarah Chen', 'Michael Torres'], 'locations': ['Riverside'], 'organizations': ['TechCorp', 'DataSystems']}) (input_keys=None)

Demo 3:
Example({'text': 'Robert Martinez founded AeroTech in Riverside, California and later moved to Nevada.', 'persons': ['Robert Martinez'], 'locations': ['Riverside, California', 'Nevada'], 'organizations': ['AeroTech']}) (input_keys={'text'})


## Step 6: Test the Optimized Version

In [ ]:

new_text = """
Dr. Sarah Martinez, Chief Technology Officer at Global Innovations Corp, traveled from 
San Francisco, California to Tokyo, Japan last Tuesday to finalize a strategic partnership 
with Sakura Technologies and their parent company, Mitsubishi Heavy Industries. During the 
three-day summit held at the Grand Hyatt Tokyo in Roppongi, she met with CEO Hiroshi Tanaka, 
CFO Yuki Nakamura, and Professor James O'Brien from MIT's Computer Science Department. 
The discussion also included representatives from the European Union's Digital Innovation Hub 
based in Brussels, Belgium, who joined via videoconference from their Luxembourg office. 

Microsoft Azure engineer Robert Chen, who previously worked at Amazon Web Services in Seattle, 
Washington, presented the technical architecture alongside his colleague Dr. Maria Garcia-Lopez 
from the Barcelona Research Center in Spain. The World Economic Forum and McKinsey & Company 
have both endorsed this collaboration, which aims to establish new data centers in Singapore, 
Dubai, and São Paulo, Brazil by Q4 2024. Former Apple executive Tim Williams, now an 
independent consultant based in Austin, Texas, moderated the sessions at the request of the 
International Technology Standards Organization (ITSO) headquartered in Geneva, Switzerland.
"""

# Compare baseline vs optimized
print("=== BASELINE ===")
baseline_result = entity_extractor_predictor(text=new_text)
print("Persons:", baseline_result.persons)
print("Locations:", baseline_result.locations)
print("Organizations:", baseline_result.organizations)

print("\n=== OPTIMIZED ===")
optimized_result = optimized_extractor(text=new_text)
print("Persons:", optimized_result.persons)
print("Locations:", optimized_result.locations)
print("Organizations:", optimized_result.organizations)

=== BASELINE ===
Persons: ['Dr. Sarah Martinez', 'Hiroshi Tanaka', 'Yuki Nakamura', "James O'Brien", 'Robert Chen', 'Maria Garcia-Lopez', 'Tim Williams']
Locations: ['San Francisco, California', 'Tokyo, Japan', 'Grand Hyatt Tokyo', 'Roppongi', 'Brussels, Belgium', 'Luxembourg', 'Seattle, Washington', 'Barcelona', 'Singapore', 'Dubai', 'São Paulo, Brazil', 'Austin, Texas', 'Geneva, Switzerland']
Organizations: ['Global Innovations Corp', 'Mitsubishi Heavy Industries', 'Sakura Technologies', 'Microsoft Azure', 'Amazon Web Services', 'MIT', "European Union's Digital Innovation Hub", 'World Economic Forum', 'McKinsey & Company', 'International Technology Standards Organization (ITSO)']

=== OPTIMIZED ===
Persons: ['Dr. Sarah Martinez', 'Hiroshi Tanaka', 'Yuki Nakamura', "Professor James O'Brien", 'Robert Chen', 'Dr. Maria Garcia-Lopez', 'Tim Williams']
Locations: ['San Francisco, California', 'Tokyo, Japan', 'Roppongi', 'Grand Hyatt Tokyo', 'Brussels', 'Luxembourg', 'Seattle, Washington', 

In [22]:


print("========= Baseline Extractor Prompt ==========")
# Run baseline extractor
_ = extractor(text=new_text)
# Inspect the prompt that was just used
lm.inspect_history(n=1)

print("\n\n========= Optimized Extractor Prompt ==========")
# Run optimized extractor
_ = optimized_extractor(text=new_text)
# Inspect the prompt that was just used (notice the few-shot examples!)
lm.inspect_history(n=1)



========= Baseline Extractor Prompt ==========




[2026-02-04T22:32:06.293782]

System message:

Your input fields are:
1. `text` (str): The text to extract entities from
Your output fields are:
1. `reasoning` (str): 
2. `persons` (str): List of person names, comma-separated
3. `locations` (str): List of location names, comma-separated
4. `organizations` (str): List of organization names, comma-separated
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## reasoning ## ]]
{reasoning}

[[ ## persons ## ]]
{persons}

[[ ## locations ## ]]
{locations}

[[ ## organizations ## ]]
{organizations}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Extract named entities (persons, locations, organizations) from the given text.


User message:

[[ ## text ## ]]

GreenTech CEO David Kim visited the National Innovation Center in Capital City to meet with 
Director John Smith and discuss ren

## Key Takeaways

1. **Declarative Programming**: Define *what* you want (signature), not *how* to prompt
2. **Automatic Prompting**: DSPy generates the prompt automatically
3. **Self-Improvement**: The optimizer learns from examples to improve performance
4. **Modularity**: Easy to swap models, add reasoning (ChainOfThought), or change optimizers

## Some things you can try next
- Try with your own text and entities, or other use cases besides entity extraction
- Experiment with different signatures (e.g., add entity types like "dates" or "products")
- Use better metrics for optimization
- Test with different LMs and compare performance